# Component 2: diet and 16S community composition (LONGIT-weeklong)

Does diet (live-microbe foods, fibre variety) relate to the 16S community composition of stool, capsule and saliva samples?
The diet logs are described in notebook `component0_diet_logs`. This notebook follows the workflow in
`diet_log_analyses/CLAUDE.md`; reusable code is in `scripts/diet_log.py`.

**Sections**
- **1. Analysis plan** (stool; approved 2026-10-18, see `PROGRESS.md`).
- Later sections: stool analyses, then capsules, then saliva.

Figures are shown here and saved to `figures/component2_diet_16S/` as PNG (300 dpi) and PDF, each with the CSV it was drawn from.

## Parameters
Everything that can change a result is set here, once. The notebook may be run from any folder: the first lines
make `diet_log_analyses/scripts` importable, which also makes the shared modules in `2-analysis_python/` importable.

In [1]:
import sys
from pathlib import Path

# make diet_log_analyses/scripts importable when the notebook is run from another folder
here = Path.cwd()
scripts = next(p for p in [here, here / "diet_log_analyses" / "scripts", here / "scripts"] if (p / "_paths.py").exists())
sys.path.insert(0, str(scripts))

import json
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import _paths            # paths: FIG_DIR, DIET_CODING; also puts 2-analysis_python on sys.path
import config as C       # shared helpers (read_excel_like_readxl, ...)
import diet_log as DL    # diet-log tables and figures for these notebooks

FIG_FOLDER = "component2_diet_16S"   # all figures of this notebook go to figures/<FIG_FOLDER>/
TOP_N = 40                           # ingredients shown in the frequency figure
TOP_N_CATEGORY = 15                  # foods shown per food class
CLASS_LEVEL = "Yes"                  # lexicon value that puts a food in a class ("Variable" is not counted)

# analysis plan (section 1; approved 2026-10-18, see PROGRESS.md). Change only with a dated deviation.
PLAN = {
    "sample_type": "Stool",               # this part of the plan; capsules and saliva get their own plans
    "window_days": 3,                     # exposure window: the 3 calendar days before the collection day
    "window_days_sensitivity": 2,         # sensitivity analysis: the 2 days before
    "min_reads": 1000,                    # stools with fewer 16S reads are excluded
    "clr_prevalence": 0.10,               # ASVs kept for CLR: present in >= 10% of stools
    "clr_pseudocount": 1.0,               # added to every count before the log (sensitivity: 0.5)
    "genus_prevalence": 0.20,             # exploratory genus models: genera in >= 20% of stools
    "n_permutations": 9999,               # PERMANOVA permutations
    "fdr_alpha": 0.05,                    # false discovery rate for Benjamini-Hochberg correction
    "seed": 20261018,                     # one seed for rarefying, permutations and simulations
}
# config.py switches matplotlib to a non-display backend; show figures inline in the notebook again
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.family": "sans-serif"})
print("figures will be saved to:", _paths.FIG_DIR / FIG_FOLDER)

figures will be saved to: /Users/lisa/Documents/GitHub/gen2-longit/diet_log_analyses/figures/component2_diet_16S


## Input versions (workflow steps 0 and 2)
The input profiler (`scripts/profile_inputs.py`) records a checksum for every input in `figures/input_profiles/manifest.json`.
This cell recomputes the checksum of the diet workbook used here. **Expected:** it matches the manifest. A mismatch means the
workbook changed since it was last profiled: rerun the profiler and the tests before trusting anything below.

In [2]:
manifest = json.loads((_paths.FIG_DIR / "input_profiles" / "manifest.json").read_text())
sha_now = hashlib.sha256(_paths.DIET_CODING.read_bytes()).hexdigest()
entry = manifest["daily_diet_coding"]
print(f"diet workbook: {_paths.DIET_CODING.name}, modified {entry['modified']}")
print(f"checksum now:          {sha_now[:16]}…")
print(f"checksum in manifest:  {entry['sha256'][:16]}…")
print("MATCH: workbook unchanged since profiling" if sha_now == entry["sha256"] else "MISMATCH: rerun profile_inputs.py and the tests")

diet workbook: daily_diet_coding.xlsx, modified 2026-10-18T21:00:48
checksum now:          de90136d6a2fdc98…
checksum in manifest:  de90136d6a2fdc98…
MATCH: workbook unchanged since profiling


# 1. Analysis plan (component 2, stool)

This section is the plan, written **before any microbiome data are analysed** (CLAUDE.md, workflow step 1). It was agreed
on 2026-10-18; each decision is dated in `PROGRESS.md`. Any later change is recorded as a dated deviation, here and in
`PROGRESS.md`. Fixing the plan first matters because a study like this allows many reasonable choices (which foods, which
days, which measure); choosing them after seeing the results makes chance findings look real.

Every statistical method is explained in plain language where it first appears. The numbers that control the analysis are
in the `PLAN` dictionary in the parameters cell at the top of the notebook and are printed at the end of this section.

## 1.1 Questions

**Q1, within a person (primary).** When a person ate more live-microbe foods, or a greater variety of fibrous foods, in the
days before a stool sample, is that sample different from the same person's other samples?

**Q2, between people (primary).** Do people who usually eat more live-microbe foods, or a greater variety of fibrous foods,
have different stool microbiomes from people who eat less?

Why both: people differ from each other in microbiome far more than one person differs from day to day. Q1 compares each
person only with themselves, so those differences cannot masquerade as a diet effect. Q2 asks the "habitual diet" question,
but has only 19 people, so it can detect only large differences.

**What to expect.** In the Sonnenburg-lab fermented-food trial, diversity rose gradually over weeks of a deliberate diet.
Here we have one ordinary week, so short-term effects may be too small to detect. A clear "no detectable effect", reported
with the smallest effect the design could have found (1.8), is a useful result.

## 1.2 Data and exclusions

- **Samples:** stool samples with 16S data from the 19 subjects with a coded diet log. Subject 11 has no diet log and is
  excluded from everything that uses it.
- **Too few reads:** stools with fewer than `PLAN["min_reads"]` (1,000) 16S reads are excluded, because diversity estimated
  from very few reads is unreliable.
- **Incomplete exposure window:** a stool is used only if every day of its exposure window (1.3) was logged. A day that was
  not logged is unknown, not "ate nothing", so a window with a gap would understate what was eaten.
- Every exclusion is counted and reported when the data are loaded; none is silent.

## 1.3 Exposures: what the person ate before each sample

**Exposure window.** For each stool, the 3 calendar days before the collection day (`PLAN["window_days"]`). Example: a stool
collected on 6 August uses 3, 4 and 5 August. Calendar days are used because only 41% of log entries have a time; for a
morning sample this covers food eaten roughly 7 to 79 hours earlier, which matches gut transit time and the speed at which
the microbiome responds to diet. Sensitivity analysis: the 2 days before (`PLAN["window_days_sensitivity"]`).

**Live-microbe dose (primary exposure 1).** The number of days in the window (0-3) on which the person logged at least one
live-microbe food (lexicon `Likely live microbes? = Yes`: kimchi, natto, yogurt). Days, not servings, because the log records
what was eaten but not how much.

**Fibre variety (primary exposure 2).** The average number of different fibrous foods per day in the window (lexicon
`Fibrous? = Yes`, including wheat). A count of different foods, not a yes/no, because every logged day has at least one.

**Sensitivity analyses** (rerun to see whether conclusions change): live-microbe dose counted as yes/no; `Variable` foods
counted as live-microbe; stated foods only (ignoring foods inferred from dishes). **Exploratory:** the broad `Fermented? = Yes`
class.

**Habitual intake for Q2.** Each person's average over all their logged days: the share of days with a live-microbe food,
and the average number of different fibrous foods per day. Sensitivity analysis for live-microbe foods: the questionnaire
answer (there is no fibre question). The diet log is the primary measure because the questionnaire agreed only loosely with
the logged week (notebook `component0_diet_logs`, section 0.9).

## 1.4 Outcomes: how the microbiome is described

**Alpha diversity** (diversity within one sample):
- **Richness:** the number of different ASVs (sequence variants) in the sample.
- **Shannon index:** combines richness with evenness. It is high when many taxa are present in similar amounts and low when
  a few taxa dominate. (Richness is the primary alpha outcome; Shannon is secondary.)
- **Rarefying.** Samples are sequenced to very different depths, and a deeper sample finds more rare taxa simply because more
  reads were drawn. Before computing alpha diversity, every sample is randomly subsampled to the same number of reads: the
  smallest read count among the included stools. A fixed seed (`PLAN["seed"]`) makes this reproducible.

**Beta diversity** (how different two samples are). Each distance below gives one number for every pair of samples, from
0 (identical) upwards.
- **Aitchison distance (primary).** Sequencing gives proportions, not amounts: if one taxon doubles, every other taxon's
  share falls even if nothing happened to it. The **CLR transform** (centred log-ratio) avoids this by describing each taxon
  relative to the sample's average taxon on a log scale: CLR = log(count) − mean of the log counts in that sample. Values
  are positive for taxa more abundant than the average taxon and negative for less abundant ones. The Aitchison distance is
  the ordinary straight-line distance between two samples' CLR values. It does not depend on sequencing depth.
  - **Zeros.** log(0) is undefined, and most ASVs are absent from most samples. So before CLR, only ASVs present in at least
    10% of stools are kept (`PLAN["clr_prevalence"]`) and 1 is added to every count (`PLAN["clr_pseudocount"]`). The choice
    of pseudocount affects the result, so sensitivity analyses use 0.5, and genus level instead of ASV level.
- **Secondary distances**, each emphasising a different kind of difference (computed with the shared, R-validated
  `config.distance()`):
  - **Bray-Curtis** (on relative abundances): differences in the abundant taxa.
  - **Weighted UniFrac** (relative abundances + phylogenetic tree): shifts in the abundant lineages.
  - **Canberra** (rarefied counts): every taxon counts equally, so rare taxa matter.
  - **Jaccard / binary** (rarefied presence/absence): the share of taxa found in only one of the two samples.
  - **Unweighted UniFrac** (rarefied presence/absence + tree): the share of the tree's branch length found in only one sample,
    i.e. whether different lineages are present.
  Presence/absence measures use rarefied counts because detecting a rare taxon depends on sequencing depth.

## 1.5 Models for Q1: comparing each person with themselves

**Splitting each exposure into "usual" and "more or less than usual".** For each stool, the exposure is split into two parts:
the person's own average over their usable stools, and the deviation from that average for this stool. Example (subject 03,
live-microbe dose): samples with doses 2, 1, 0, 1, 3 have an average of 1.4 and deviations +0.6, −0.4, −1.4, −0.4, +1.6.
Q1 is answered by the deviation only. Without the split, one coefficient would mix "people who eat more yogurt differ" with
"a person changes after eating yogurt", which can even point in opposite directions (Curran & Bauer 2011).

**Linear mixed model, for richness and Shannon.**
`richness ~ live deviation + live average + fibre deviation + fibre average + log(read depth) + study day + (1 | subject)`
- A **linear model** (regression) estimates how much the outcome changes per unit of each variable, holding the others fixed.
  Here, for example, "change in richness per extra live-microbe day in the window".
- `(1 | subject)` is a **random intercept**: each person gets their own baseline level of richness. This is needed because
  samples from the same person resemble each other; treating 92 samples from 19 people as 92 independent observations would
  make the results look far more certain than they are (pseudo-replication).
- **Covariates** (variables held fixed so they cannot create a false diet effect): read depth (deeper sequencing finds
  more taxa) and study day (the microbiome and the diet can both drift over the week). Bristol stool score is added only in
  a sensitivity analysis, because stool consistency may itself be changed by diet; adjusting for it could remove part of a
  real effect (a **mediator**).
- **What is reported:** for each deviation term, the estimate (e.g. "+X ASVs per extra live-microbe day"), its **95%
  confidence interval** (the range of effects consistent with the data) and a **p-value** (how surprising an estimate this
  large would be if there were truly no effect). Fitted with Python `statsmodels` (REML) and cross-checked with R `nlme`.

**PERMANOVA, for Aitchison distance (and the secondary distances)** (Anderson 2001).
- PERMANOVA asks whether samples with higher exposure sit in a different part of "distance space" than samples with lower
  exposure. Its statistic (pseudo-F) compares how far apart samples are across exposure levels with how far apart they are
  overall; R² is the share of the variation in distances explained by the term.
- **Permutation test.** The p-value comes from shuffling: the exposure values are shuffled many times
  (`PLAN["n_permutations"]`), the statistic is recomputed each time, and the p-value is the share of shuffles that give a
  statistic at least as large as the real one. No assumption about the shape of the data is needed.
- **Shuffling only within each person** (and including subject as a term). Shuffling across people would mostly show that
  people differ, which we already know. Shuffling within each person keeps every comparison "person against themselves".
- Computed in Python and cross-checked with R `vegan::adonis2`.

## 1.6 Models for Q2: comparing people

With one habitual value per person, Q2 uses **one value per person for the outcome too** (their average richness; their
average CLR profile), so the sample size is 19, not 92. Using all samples would again treat one person's samples as
independent evidence.
- **Spearman rank correlation**, for richness: ranks the 19 people by habitual intake and by average richness and measures
  how well the two rankings agree (from −1, opposite, through 0, unrelated, to +1, identical). Ranks make it insensitive to
  how intake is scaled and to single extreme values.
- **PERMANOVA across people**, for Aitchison distance: each person is one point (their average CLR profile); shuffling is free
  across people because each appears once.

## 1.7 Which tests count: primary, secondary, exploratory

The more tests are run, the more likely some come out "significant" by chance (at p < 0.05, about 1 in 20 tests of a true
null). So tests are fixed in three tiers before any are run.
- **Primary (8 tests):** Q1 and Q2, each on richness and on Aitchison distance, for live-microbe foods and for fibre. These
  are the main results. They are corrected together with the **Benjamini-Hochberg** false discovery rate (FDR): p-values are
  adjusted so that, among the results called significant, the expected share of false positives is at most 5%
  (`PLAN["fdr_alpha"]`).
- **Secondary:** Shannon; the five secondary distances; the between-person terms of the mixed model. Reported together, with
  FDR correction across the set, and read as a pattern (which kind of change, if any) rather than as separate findings.
- **Sensitivity analyses** (1.3, 1.4, 1.5): not new findings; they show whether the primary results survive other reasonable
  choices.
- **Exploratory (labelled as such):** the broad fermented-food class; individual genera present in at least 20% of stools,
  one CLR mixed model each with FDR correction. *Streptococcus* and *Lactobacillus* (cultures in yogurt) are named in
  advance as genera expected to respond to live-microbe foods.

All effects are reported with confidence intervals, not only p-values.

## 1.8 Checks and method tests (before and during the analysis)

**Method tests, before any real data** (CLAUDE.md, "Testing new statistical methods"; saved in `scripts/tests/`):
- **Simulated data** shaped like ours (19 subjects, 4-10 stools each, similar sparsity). With a **planted effect**, each
  method must find it; with **no effect**, about 5% of p-values should fall below 0.05 and p-values should be spread evenly.
  The simulations also give the **power**: the smallest effect these data could detect, reported with any null result.
- **A tiny case worked out by hand** for the CLR, the Aitchison distance and the PERMANOVA statistic.
- **A second implementation**: R `nlme` for the mixed model, R `vegan::adonis2` for PERMANOVA.
- **A deliberate break** (e.g. shuffling subject labels) to confirm the tests notice.

**Checks on the real data** (workflow step 3), each with its expected result stated before running:
- Row counts at every join and exclusion (expected: about 92 usable stools from 19 subjects).
- Shuffling the exposure within each person gives no effect.
- **Negative control:** diet in the 3 days *after* the stool cannot have caused it, so it should show no effect.
- **Positive control:** subject identity should explain a large share of the variation in composition.
- By-hand trace of 3-4 random stools from the sample sheet to their exposure values and reads.

## 1.9 Order of work

1. Method tests on simulated data (1.8), with results explained in chat before going on.
2. Load and join the stool data, with counts at every step; exposures per stool; checks (1.8).
3. Primary analyses (1.7), then secondary and sensitivity analyses, then exploratory genus models.
4. Every figure inspected; the whole notebook rerun from scratch; results reported with the cell they come from; independent
   review (workflow step 8); `PROGRESS.md` updated.

The analysis parameters used everywhere below:

In [3]:
plan_table = pd.DataFrame({"parameter": list(PLAN), "value": [str(v) for v in PLAN.values()]})
print(plan_table.to_string(index=False))

              parameter    value
            sample_type    Stool
            window_days        3
window_days_sensitivity        2
              min_reads     1000
         clr_prevalence      0.1
        clr_pseudocount      1.0
       genus_prevalence      0.2
         n_permutations     9999
              fdr_alpha     0.05
                   seed 20261018
